# Package v0: merged AI job-risk table

**Goal:** turn the Milestone 1 outputs into one documented, shareable package: a clean table with scores,
percentile ranks and provenance flags, a no-drop version with every occupation, and a data dictionary.

**Reads:** `merged_ai_risk_v0.csv` and `coverage_report_v0.csv` (Tasks 4-5), `percentile_ranks_v0.csv`
(Task 7), and the interim tables. For the QA trace it also rereads the raw files. **Writes** to
`data/processed/` (earlier files are not overwritten):

| File | Contents |
|---|---|
| `ai_risk_v0.csv` | 693 occupations scored by all three indices: scores, percentile ranks, provenance flags |
| `ai_risk_v0_all_occupations.csv` | All 808 occupations scored by *any* index, nothing dropped (per the team's don't-delete note) |
| `ai_risk_v0_data_dictionary.csv` | One row per column: meaning, source, units |
| `ai_risk_v0.xlsx` | All of the above plus the coverage report, one sheet each, with a README sheet |

**Selected rules**
- **No score is recomputed.** Scores come from Task 4 and ranks from Task 7; this notebook only selects,
  renames and documents. The QA trace in §5 checks that against the raw sources.
- **Two tables, not one.** `ai_risk_v0` is the three-way intersection, which is what cross-index
  comparisons need. `all_occupations` keeps every occupation with missing scores left blank, never zero.
- **Flag, don't drop.** Averaged or copied scores keep their provenance columns, plus `any_provenance_flag`.
- **Titles come from the official BLS SOC 2018 list**, with the crosswalk's `(#)` / `(##)` markers removed.

**Run order to reproduce:** `Milestone_1.ipynb` (Tasks 1-5) → `task7_percentile_ranks.ipynb` → this notebook.

## Setup

Works in **Google Colab** and **locally**:
- **Colab:** mounts Google Drive and reads from the team folder
  `MyDrive/BTT Fall AI Studio (swytch 1b)/data/` (the same folder `Milestone_1.ipynb` uses). If the folder was
  shared with you, add a shortcut to it in *My Drive* first. Any file missing from Drive is downloaded
  from the GitHub repo instead, so the notebook still runs without Drive access.
- **Locally:** reads `data/` from the repo (run from anywhere inside the repo with the project `.venv`).

In [1]:
import subprocess
import sys
import tempfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 70)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/BTT Fall AI Studio (swytch 1b)")
else:
    # Repo root = nearest folder (this one or above) that contains data/, so any notebooks/ subfolder works
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), Path.cwd())

DATA = ROOT / "data"
# Fallback source for files not found under DATA (switch the branch to main once this is merged)
GITHUB_DATA = "https://raw.githubusercontent.com/Break-Through-Tech/Swytch-1B-do-ai-job-risk-indices-agree/kamsi/data"
print("data folder:", DATA, "(found)" if DATA.exists() else "(not found, using GitHub)")


def data_file(rel):
    """Path to data/<rel>: from Drive / the repo if present, else downloaded once from GitHub."""
    path = DATA / rel
    if path.exists():
        return path
    cached = Path(tempfile.gettempdir()) / "swytch_data" / rel
    if not cached.exists():
        cached.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading {rel} from GitHub")
        urllib.request.urlretrieve(f"{GITHUB_DATA}/{rel}", cached)
    return cached


OUT = (ROOT if ROOT.exists() else Path.cwd()) / "notebooks" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

MAJOR_GROUPS = {
    "11": "Management", "13": "Business & Financial", "15": "Computer & Mathematical",
    "17": "Architecture & Engineering", "19": "Life, Physical & Social Sci.",
    "21": "Community & Social Service", "23": "Legal", "25": "Education & Library",
    "27": "Arts, Design & Media", "29": "Healthcare Practitioners", "31": "Healthcare Support",
    "33": "Protective Service", "35": "Food Preparation & Serving", "37": "Building & Grounds",
    "39": "Personal Care & Service", "41": "Sales", "43": "Office & Admin Support",
    "45": "Farming, Fishing & Forestry", "47": "Construction & Extraction",
    "49": "Installation & Repair", "51": "Production", "53": "Transportation & Material Moving",
    "55": "Military",
}


def boxplot_by_group(df, code_col, value_col, title):
    """Horizontal boxplot of value_col per SOC major group, sorted by median."""
    groups = df.assign(mg=df[code_col].str[:2]).groupby("mg")[value_col]
    order = groups.median().sort_values().index
    data = [groups.get_group(g) for g in order]
    fig, ax = plt.subplots(figsize=(8, 7))
    try:
        ax.boxplot(data, orientation="horizontal")  # matplotlib >= 3.10
    except TypeError:
        ax.boxplot(data, vert=False)
    ax.set_yticks(range(1, len(order) + 1), [f"{g} {MAJOR_GROUPS[g]}" for g in order])
    ax.set_xlabel(value_col)
    ax.set_title(title, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()


def load_crosswalk():
    """BLS SOC 2010 -> 2018 crosswalk with short column names."""
    cw = pd.read_excel(data_file("raw/soc_2010_to_2018_crosswalk.xlsx"), header=8, dtype=str)
    cw.columns = ["soc10", "title10", "soc18", "title18"]
    return cw.apply(lambda s: s.str.strip())

data folder: C:\Users\Kamsi\Downloads\code\Swytch-1B-do-ai-job-risk-indices-agree\data (found)


## 1. Inputs

In [2]:
# Where task7_percentile_ranks.ipynb saved its output (Drive / repo, or ./output without Drive)
out_dir = DATA / "processed" if DATA.exists() else Path.cwd() / "output"
ranks_path = out_dir / "percentile_ranks_v0.csv"

merged = pd.read_csv(data_file("processed/merged_ai_risk_v0.csv"))
ranks = pd.read_csv(ranks_path if ranks_path.exists() else data_file("processed/percentile_ranks_v0.csv"))
coverage = pd.read_csv(data_file("processed/coverage_report_v0.csv"))
elo6 = pd.read_csv(data_file("interim/eloundou_soc6.csv")).rename(columns={"soc6": "soc"})
aioe_x = pd.read_csv(data_file("interim/aioe_soc2018.csv")).rename(columns={"soc2018": "soc"})
fo_x = pd.read_csv(data_file("interim/frey_osborne_soc2018.csv")).rename(columns={"soc2018": "soc"})

cw = load_crosswalk()
titles18 = cw.drop_duplicates("soc18").set_index("soc18").title18.str.replace(r"\s*\(#+\)$", "", regex=True)

for name, df in [("merged", merged), ("ranks", ranks), ("coverage", coverage),
                 ("Eloundou", elo6), ("AIOE", aioe_x), ("Frey-Osborne", fo_x)]:
    print(f"{name:<13} {df.shape}")

merged        (693, 26)
ranks         (693, 10)
coverage      (115, 5)
Eloundou      (798, 10)
AIOE          (800, 7)
Frey-Osborne  (699, 7)


## 2. Build the v0 table
One row per SOC 2018 occupation. Redundant columns from the join (per-source titles, `has_*` flags that
are all true, `n_sources`) are replaced by one clean title and a combined provenance flag.

In [3]:
v0 = merged.merge(ranks, on="soc", how="left", validate="one_to_one")
v0 = v0.assign(
    title=v0.soc.map(titles18),
    major_group=v0.soc.str[:2],
    major_group_name=v0.soc.str[:2].map(MAJOR_GROUPS),
    frey_osborne_probability=v0.probability,
    eloundou_n_onet_children=v0.n_children,
    eloundou_onet_codes=v0.child_codes,
)
v0["any_provenance_flag"] = (
    (v0.eloundou_n_onet_children > 1) | v0.aioe_is_split | v0.aioe_is_merge | v0.fo_is_split | v0.fo_is_merge
)

COLUMNS = (
    ["soc", "title", "major_group", "major_group_name"]
    + ["dv_rating_alpha", "dv_rating_beta", "dv_rating_gamma",
       "human_rating_alpha", "human_rating_beta", "human_rating_gamma",
       "AIOE", "frey_osborne_probability"]
    + [c for c in ranks.columns if c.startswith("pct_")]
    + ["eloundou_n_onet_children", "eloundou_onet_codes",
       "aioe_n_2010_parents", "aioe_parent_soc2010_codes", "aioe_is_split", "aioe_is_merge",
       "fo_n_2010_parents", "fo_parent_soc2010_codes", "fo_is_split", "fo_is_merge", "any_provenance_flag"]
)
v0 = v0[COLUMNS].sort_values("soc").reset_index(drop=True)
print(v0.shape)
v0.head()

(693, 32)


,soc,title,major_group,major_group_name,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma,AIOE,frey_osborne_probability,pct_eloundou_beta,pct_eloundou_beta_human,pct_aioe,pct_frey_osborne,pct_full_eloundou_beta,pct_full_eloundou_beta_human,pct_full_aioe,pct_full_frey_osborne,pct_spread,eloundou_n_onet_children,eloundou_onet_codes,aioe_n_2010_parents,aioe_parent_soc2010_codes,aioe_is_split,aioe_is_merge,fo_n_2010_parents,fo_parent_soc2010_codes,fo_is_split,fo_is_merge,any_provenance_flag
0,11-1011,Chief Executives,11,Management,0.133333,0.507778,0.882222,0.117778,0.369444,0.621111,1.334246,0.015,79.80,65.66,92.06,9.74,79.32,63.28,91.00,9.66,82.32,2.0,11-1011.00;11-1011.03,1.0,11-1011,False,False,1.0,11-1011,False,False,True
1,11-1021,General and Operations Managers,11,Management,0.000000,0.480769,0.961538,0.115385,0.384615,0.653846,0.574877,0.160,73.88,67.32,66.09,28.14,72.68,65.48,62.75,28.04,45.74,1.0,11-1021.00,1.0,11-1021,False,False,1.0,11-1021,False,False,False
2,11-2011,Advertising and Promotions Managers,11,Management,0.000000,0.476744,0.953488,0.255814,0.546512,0.837209,1.294387,0.039,73.30,89.18,89.90,17.97,71.68,88.47,87.62,17.95,71.93,1.0,11-2011.00,1.0,11-2011,False,False,1.0,11-2011,False,False,False
3,11-2021,Marketing Managers,11,Management,0.062500,0.500000,0.937500,0.218750,0.578125,0.937500,1.315032,0.014,78.07,91.63,90.91,8.73,77.51,91.10,88.88,8.66,82.18,1.0,11-2021.00,1.0,11-2021,False,False,1.0,11-2021,False,False,False
4,11-2022,Sales Managers,11,Management,0.000000,0.450000,0.900000,0.100000,0.483333,0.866667,1.266280,0.013,69.70,80.23,87.45,8.15,67.29,79.57,85.25,8.08,79.29,1.0,11-2022.00,1.0,11-2022,False,False,1.0,11-2022,False,False,False


## 3. All-occupations table (nothing dropped)
Every occupation scored by at least one index, with missing scores left empty and the reason recorded.

In [4]:
scores = (elo6[["soc"] + [c for c in elo6.columns if "rating" in c]]
          .merge(aioe_x[["soc", "AIOE"]], on="soc", how="outer")
          .merge(fo_x[["soc", "probability"]].rename(columns={"probability": "frey_osborne_probability"}),
                 on="soc", how="outer"))
scores["has_eloundou"] = scores.dv_rating_beta.notna()
scores["has_aioe"] = scores.AIOE.notna()
scores["has_frey_osborne"] = scores.frey_osborne_probability.notna()
scores["n_sources"] = scores[["has_eloundou", "has_aioe", "has_frey_osborne"]].sum(axis=1)
scores["in_v0"] = scores.soc.isin(v0.soc)
scores = scores.merge(coverage[["soc", "reason"]], on="soc", how="left")
scores["reason"] = scores.reason.fillna("in_v0")
scores.insert(1, "title", scores.soc.map(titles18))
scores.insert(2, "major_group", scores.soc.str[:2])
all_occ = scores.sort_values("soc").reset_index(drop=True)
print(all_occ.shape)
print(all_occ.reason.value_counts().to_string())

(808, 17)
reason
in_v0                                     693
missing_from_frey_osborne                  97
missing_from_aioe_and_frey_osborne          8
missing_from_eloundou                       6
missing_from_eloundou_and_frey_osborne      4


## 4. Data dictionary

In [5]:
DESCRIPTIONS = {
    "soc": ("SOC 2018 detailed occupation code", "BLS SOC 2018", "code"),
    "title": ("SOC 2018 occupation title (BLS crosswalk, split/merge markers removed)", "BLS crosswalk", "text"),
    "major_group": ("First two digits of the SOC code", "derived", "code"),
    "major_group_name": ("SOC major group name", "derived", "text"),
    "dv_rating_alpha": ("Eloundou alpha (E1), GPT-4 rater: share of tasks directly exposed to LLMs", "Eloundou et al. (occ_level.csv), averaged over O*NET children", "share 0-1"),
    "dv_rating_beta": ("Eloundou beta (E1 + 0.5 E2), GPT-4 rater: headline exposure measure", "Eloundou et al.", "share 0-1"),
    "dv_rating_gamma": ("Eloundou gamma (E1 + E2), GPT-4 rater: exposure including LLM-powered tools", "Eloundou et al.", "share 0-1"),
    "human_rating_alpha": ("Eloundou alpha, human raters", "Eloundou et al.", "share 0-1"),
    "human_rating_beta": ("Eloundou beta, human raters", "Eloundou et al.", "share 0-1"),
    "human_rating_gamma": ("Eloundou gamma, human raters", "Eloundou et al.", "share 0-1"),
    "AIOE": ("AI Occupational Exposure; mean of 2010 parents where SOC codes merged", "Felten, Raj & Seamans (AIOE_DataAppendix.xlsx, Appendix A)", "z-score"),
    "frey_osborne_probability": ("Probability of computerisation; mean of 2010 parents where SOC codes merged", "Frey & Osborne (2013) appendix", "probability 0-1"),
    "pct_spread": ("Max minus min of pct_eloundou_beta, pct_aioe, pct_frey_osborne: cross-index disagreement", "task7_percentile_ranks.ipynb", "percentile points"),
    "eloundou_n_onet_children": ("Number of O*NET occupations averaged into this SOC code", "Milestone 1 Task 2", "count"),
    "eloundou_onet_codes": ("The O*NET-SOC codes averaged (;-separated)", "Milestone 1 Task 2", "codes"),
    "aioe_n_2010_parents": ("Number of SOC 2010 codes mapped to this code for AIOE", "Milestone 1 Task 3a", "count"),
    "aioe_parent_soc2010_codes": ("SOC 2010 codes mapped to this code for AIOE (;-separated)", "Milestone 1 Task 3a", "codes"),
    "aioe_is_split": ("AIOE score copied from a 2010 code that split into several 2018 codes", "Milestone 1 Task 3a", "bool"),
    "aioe_is_merge": ("AIOE score averaged over several 2010 codes that merged", "Milestone 1 Task 3a", "bool"),
    "fo_n_2010_parents": ("Number of SOC 2010 codes mapped to this code for Frey-Osborne", "Milestone 1 Task 3b", "count"),
    "fo_parent_soc2010_codes": ("SOC 2010 codes mapped to this code for Frey-Osborne (;-separated)", "Milestone 1 Task 3b", "codes"),
    "fo_is_split": ("Frey-Osborne score copied from a 2010 code that split", "Milestone 1 Task 3b", "bool"),
    "fo_is_merge": ("Frey-Osborne score averaged over 2010 codes that merged", "Milestone 1 Task 3b", "bool"),
    "any_provenance_flag": ("True if any score was averaged or copied (see the flags above)", "derived", "bool"),
    "has_eloundou": ("Occupation has an Eloundou score", "derived", "bool"),
    "has_aioe": ("Occupation has an AIOE score", "derived", "bool"),
    "has_frey_osborne": ("Occupation has a Frey-Osborne score", "derived", "bool"),
    "n_sources": ("Number of indices scoring the occupation", "derived", "count 1-3"),
    "in_v0": ("Occupation is in ai_risk_v0.csv (scored by all three)", "derived", "bool"),
    "reason": ("Why the occupation is not in v0 (or 'in_v0')", "coverage_report_v0.csv", "text"),
}
PCT_LABELS = {"eloundou_beta": "Eloundou beta (GPT-4)", "eloundou_beta_human": "Eloundou beta (human)",
              "aioe": "AIOE", "frey_osborne": "Frey-Osborne probability"}
for key, label in PCT_LABELS.items():
    DESCRIPTIONS[f"pct_{key}"] = (f"Percentile rank of {label} among the 693 v0 occupations (higher = more exposed / at risk)",
                                  "task7_percentile_ranks.ipynb", "percentile 0-100")
    DESCRIPTIONS[f"pct_full_{key}"] = (f"Percentile rank of {label} among all occupations that index scores",
                                       "task7_percentile_ranks.ipynb", "percentile 0-100")

rows = []
for table, df in [("ai_risk_v0", v0), ("ai_risk_v0_all_occupations", all_occ)]:
    for col in df.columns:
        desc, source, units = DESCRIPTIONS[col]
        rows.append({"table": table, "column": col, "description": desc, "source": source,
                     "units": units, "dtype": str(df[col].dtype)})
dictionary = pd.DataFrame(rows)
print(dictionary.groupby("table").size().to_string())
dictionary[dictionary.table == "ai_risk_v0"]

table
ai_risk_v0                    32
ai_risk_v0_all_occupations    17


,table,column,description,source,units,dtype
0,ai_risk_v0,soc,SOC 2018 detailed occupation code,BLS SOC 2018,code,str
1,ai_risk_v0,title,"SOC 2018 occupation title (BLS crosswalk, split/merge markers remo...",BLS crosswalk,text,str
2,ai_risk_v0,major_group,First two digits of the SOC code,derived,code,str
3,ai_risk_v0,major_group_name,SOC major group name,derived,text,str
4,ai_risk_v0,dv_rating_alpha,"Eloundou alpha (E1), GPT-4 rater: share of tasks directly exposed ...","Eloundou et al. (occ_level.csv), averaged over O*NET children",share 0-1,float64
5,ai_risk_v0,dv_rating_beta,"Eloundou beta (E1 + 0.5 E2), GPT-4 rater: headline exposure measure",Eloundou et al.,share 0-1,float64
6,ai_risk_v0,dv_rating_gamma,"Eloundou gamma (E1 + E2), GPT-4 rater: exposure including LLM-powe...",Eloundou et al.,share 0-1,float64
7,ai_risk_v0,human_rating_alpha,"Eloundou alpha, human raters",Eloundou et al.,share 0-1,float64
8,ai_risk_v0,human_rating_beta,"Eloundou beta, human raters",Eloundou et al.,share 0-1,float64
9,ai_risk_v0,human_rating_gamma,"Eloundou gamma, human raters",Eloundou et al.,share 0-1,float64


## 5. QA: trace five occupations back to the raw sources

Five occupations, chosen to cover each join situation. For each, the headline scores are recomputed
from the **raw** files (not the interim ones) and compared with the packaged row:

1. **43-9021 Data Entry Keyers:** clean 1:1 match on every index.
2. **11-1011 Chief Executives:** Eloundou average of two O\*NET occupations.
3. **11-2032 Public Relations Managers:** AIOE and Frey-Osborne scores copied from a 2010 code that split.
4. **13-1082 Project Management Specialists:** three 2010 codes merged for AIOE, two for Frey-Osborne.
5. **15-1299 Computer Occupations, All Other:** Eloundou average of nine O\*NET occupations, plus an AIOE merge.

Recompute rule, matching Milestone 1: Eloundou = mean over O\*NET codes starting with the SOC code;
AIOE and Frey-Osborne = mean over the 2010 codes the BLS crosswalk maps to the 2018 code.

In [6]:
import re
try:
    import pdfplumber
except ImportError:  # not preinstalled on Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"])
    import pdfplumber
QA_CODES = ["43-9021", "11-1011", "11-2032", "13-1082", "15-1299"]

elo_raw = pd.read_csv(data_file("raw/occ_level.csv"))
aioe_raw = pd.read_excel(data_file("raw/AIOE_DataAppendix.xlsx"), sheet_name="Appendix A")
FO_ROW = re.compile(r"^(\d+)\.\s+([\d.]+)\s+(?:(\d)\s+)?(\d{2}-\d{4})\s+(.+)$")
fo_rows = []
with pdfplumber.open(data_file("raw/The_Future_of_Employment.pdf")) as pdf:
    for page in pdf.pages[56:]:
        for line in (page.extract_text() or "").split("\n"):
            m = FO_ROW.match(line.strip())
            if m:
                fo_rows.append((m.group(4), float(m.group(2))))
fo_raw = pd.DataFrame(fo_rows, columns=["soc2010", "probability"])

trace = []
for soc in QA_CODES:
    parents = cw.loc[cw.soc18 == soc, "soc10"].unique()
    elo_kids = elo_raw[elo_raw["O*NET-SOC Code"].str[:7] == soc]
    expected = {
        "dv_rating_beta": elo_kids.dv_rating_beta.mean(),
        "AIOE": aioe_raw.loc[aioe_raw["SOC Code"].isin(parents), "AIOE"].mean(),
        "frey_osborne_probability": fo_raw.loc[fo_raw.soc2010.isin(parents), "probability"].mean(),
    }
    row = v0.set_index("soc").loc[soc]
    for col, exp in expected.items():
        trace.append({"soc": soc, "title": row.title, "score": col, "O*NET children": len(elo_kids),
                      "2010 parents": ";".join(parents), "recomputed from raw": exp, "packaged": row[col]})
trace = pd.DataFrame(trace)
trace["match"] = (trace["recomputed from raw"] - trace["packaged"]).abs() < 1e-9
trace.round(4)

,soc,title,score,O*NET children,2010 parents,recomputed from raw,packaged,match
0,43-9021,Data Entry Keyers,dv_rating_beta,1,43-9021,0.8929,0.8929,True
1,43-9021,Data Entry Keyers,AIOE,1,43-9021,0.3108,0.3108,True
2,43-9021,Data Entry Keyers,frey_osborne_probability,1,43-9021,0.9900,0.9900,True
3,11-1011,Chief Executives,dv_rating_beta,2,11-1011,0.5078,0.5078,True
4,11-1011,Chief Executives,AIOE,2,11-1011,1.3342,1.3342,True
5,11-1011,Chief Executives,frey_osborne_probability,2,11-1011,0.0150,0.0150,True
6,11-2032,Public Relations Managers,dv_rating_beta,1,11-2031,0.5500,0.5500,True
7,11-2032,Public Relations Managers,AIOE,1,11-2031,1.2937,1.2937,True
8,11-2032,Public Relations Managers,frey_osborne_probability,1,11-2031,0.0150,0.0150,True
9,13-1082,Project Management Specialists,dv_rating_beta,1,11-9199;13-1199;15-1199,0.5500,0.5500,True


In [7]:
# Stop before exporting if any trace failed
assert trace.match.all(), trace[~trace.match]
print(f"QA trace passed: {len(trace)} values across {len(QA_CODES)} occupations match the raw sources")

QA trace passed: 15 values across 5 occupations match the raw sources


## 6. Validation

In [8]:
assert len(v0) == len(merged) and v0.soc.is_unique
assert v0.drop(columns=["eloundou_onet_codes", "aioe_parent_soc2010_codes", "fo_parent_soc2010_codes"]).notna().all().all()
assert v0.title.notna().all(), "every code should have a SOC 2018 title"
for col, src in [("dv_rating_beta", "dv_rating_beta"), ("AIOE", "AIOE"), ("frey_osborne_probability", "probability")]:
    assert (v0.set_index("soc")[col] - merged.set_index("soc")[src]).abs().max() == 0

assert all_occ.soc.is_unique and len(all_occ) == len(v0) + len(coverage)
assert set(all_occ.loc[all_occ.in_v0, "soc"]) == set(v0.soc)
assert (all_occ.in_v0 == (all_occ.n_sources == 3)).all()
print("all checks passed")
print(f"v0: {len(v0)} occupations | all_occupations: {len(all_occ)} | flagged rows in v0: {v0.any_provenance_flag.sum()}")

all checks passed
v0: 693 occupations | all_occupations: 808 | flagged rows in v0: 105


## 7. Write the package

In [9]:
out_dir.mkdir(parents=True, exist_ok=True)

readme = pd.DataFrame({"README": [
    "AI job-risk indices, merged on SOC 2018 (Swytch 1B, Milestone 1, v0)",
    "",
    "Sheets:",
    f"  ai_risk_v0: {len(v0)} occupations scored by all three indices (Eloundou et al., AIOE, Frey & Osborne), with percentile ranks and provenance flags",
    f"  all_occupations: all {len(all_occ)} occupations scored by any index; missing scores left blank, reason recorded",
    f"  coverage_report: the {len(coverage)} occupations not in ai_risk_v0 and which index is missing",
    "  data_dictionary: meaning, source and units of every column",
    "",
    "Scales differ (share / z-score / probability): compare indices using the pct_ columns only.",
    "Known limitation: Frey-Osborne uses some non-SOC-2010 codes, so education and healthcare",
    "occupations are under-represented in ai_risk_v0 (see eda_processed.ipynb).",
    "Built by notebooks/package_v0.ipynb.",
]})

v0.to_csv(out_dir / "ai_risk_v0.csv", index=False)
all_occ.to_csv(out_dir / "ai_risk_v0_all_occupations.csv", index=False)
dictionary.to_csv(out_dir / "ai_risk_v0_data_dictionary.csv", index=False)
with pd.ExcelWriter(out_dir / "ai_risk_v0.xlsx") as xl:
    readme.to_excel(xl, sheet_name="README", index=False)
    v0.to_excel(xl, sheet_name="ai_risk_v0", index=False)
    all_occ.to_excel(xl, sheet_name="all_occupations", index=False)
    coverage.to_excel(xl, sheet_name="coverage_report", index=False)
    dictionary.to_excel(xl, sheet_name="data_dictionary", index=False)

for f in sorted(out_dir.glob("ai_risk_v0*")):
    print(f"{f.name:<40} {f.stat().st_size / 1024:8.1f} KB")

ai_risk_v0.csv                              200.4 KB
ai_risk_v0.xlsx                             214.4 KB
ai_risk_v0_all_occupations.csv              142.7 KB
ai_risk_v0_data_dictionary.csv                6.3 KB


In [10]:
# Round-trip check: the saved files read back identically
back = pd.read_csv(out_dir / "ai_risk_v0.csv")
assert back.shape == v0.shape and (back.columns == v0.columns).all()
xl = pd.ExcelFile(out_dir / "ai_risk_v0.xlsx")
print("sheets:", xl.sheet_names)
assert xl.parse("ai_risk_v0").shape == v0.shape
print("round-trip OK")

sheets: ['README', 'ai_risk_v0', 'all_occupations', 'coverage_report', 'data_dictionary']


round-trip OK

## Result and retained limitations

- **`ai_risk_v0`:** 693 SOC 2018 occupations with Eloundou (alpha/beta/gamma, GPT-4 and human), AIOE and
  Frey-Osborne scores, percentile ranks computed two ways, and provenance flags for every averaged or
  copied score. Five occupations were traced back to the raw files before export.
- **`all_occupations`:** 808 occupations, none dropped. Use it when a question does not need all three indices.
- **Headline so far:** the two AI-exposure indices agree (Eloundou beta vs. AIOE, Spearman ≈ 0.86), while
  Frey-Osborne's automation risk is weakly *negatively* related to both.
- **Retained limitations**
  - Frey-Osborne coverage: 97 occupations, mostly postsecondary teachers and physicians, are missing
    because the paper uses broad or SOC 2000 codes. Expanding the broad codes would add 57; until then v0
    under-represents education and healthcare (see `eda_processed.ipynb`).
  - AIOE's broad code `19-1020` Biologists is not yet mapped to its four detailed codes.
  - Averaged and copied scores use a simple unweighted mean, which is a v0 assumption. "All Other" codes and
    merged occupations are the least reliable; filter on `any_provenance_flag` for sensitivity checks.

This completes the v0 package for Milestone 1. It does not test agreement statistically or model anything;
that is the next milestone's work.